# CDCR Facility Heat Risk Index — v0.2

Computes a facility-level heat risk index for 31 CDCR state prisons following the Ovienmhada (2024) / VCP environmental risk framework.

**Risk = 0.25H + 0.25E + 0.50V** (additive, vulnerability double-weighted)

All sub-components are min-max normalized 0–1 before averaging within each component (the hazard temperature indicators use max-normalization — see §3). Components are combined additively with weights 0.25 (Hazard), 0.25 (Exposure), 0.50 (Vulnerability), then normalized 0–100 cross-period by **max-normalization** (current and mid-century share the same cross-period maximum as denominator, with no minimum subtracted, so the two periods stay comparable and no facility is forced to a false 0 — see §6).

Vulnerability receives double weight because cooling in prisons is controlled by staff who, as Brunn et al. (2025) document, withhold AC, water, and shade to punish and retaliate. The index answers "where are people most at risk if cooling fails?" The additive form also prevents a facility with full mechanical AC from scoring zero risk; a multiplicative model would zero out CHCF, which has the highest vulnerability in the system but no indoor heat days.

**Version:** v0.2 — the hazard component was rebuilt on a LOCA2-CA daily extraction, joined per facility rather than by tract centroid. Daytime heat is a 50/50 blend of a facility-relative and an absolute (90°F) threshold; warm nights are a relative P95 count; AQI enters as a multiplicative modifier. Exposure, vulnerability, weights, and periods are unchanged from v0.1. **v0.3** changes only the final combination step: the cross-period risk score switched from min-max to max-normalization (removing the false 0 floor, e.g. CEN), and risk categories are now Jenks-classified per period rather than by shared mid-century breaks. See the changelog in `analysis/README.md`.

## Components

| Component | Weight | Sub-components | Source |
|---|---|---|---|
| **Hazard** | 0.25 | Hot days (50/50 blend of `loca2_days_over_avg_plus10` relative + `loca2_days_over_90` absolute) + warm nights (`loca2_nights_over_p95`), max-normalized and averaged, × AQI modifier (1 + 0.30·AQI_norm/100) | `data/hazards/heat_air_hazard.csv` via `cdcr_code` |
| **Exposure** | 0.25 | `days_indoor_above_78f_2025`, `ratio_indoor_to_outdoor`, `uhi_normalized`, `1 - pct_hu_mechanical` | `data/cdcr/indoor_outdoor_heat_2025.csv` + `data/cdcr/cdcr_facilities.csv` |
| **Vulnerability** | 0.50 | Medical acuity (P1+P2+medium), age >50, mental health (EOP), disability (DPP), race/POC, % female | `data/cdcr/cdcr_facilities.csv` |

**Note on facility coverage:** 31 of 34 state prisons have indoor exposure data. CAC, CVSP, and FWF are excluded (no indoor/outdoor heat model data available).

**Note on UHI nulls:** CCI and PVSP have no Benz & Burney (2021) UHI data — their tracts were classified as undeveloped. Imputed with system mean across 31 facilities.

**Note on the AC sub-indicator:** uses `pct_hu_mechanical` (fraction of housing units with mechanical AC) from the CDCR Air Cooling Pilot Supplemental Report (Jan 2026, as of Dec 2025) — the newest complete per-facility source. It replaces the older, incomplete Reuters FOIA `pct_units_refrigeration`, which overstated AC at 16 of 31 facilities (e.g. CIM 100% vs the report's ~43%).

**Note on PBSP ratio outlier:** PBSP (Pelican Bay, Crescent City coast) has `ratio_indoor_to_outdoor` = 15.75, driven by very few outdoor 78°F days (~4) in that coastal climate. This is physically plausible but will score PBSP at 1.0 on this sub-component. Flagged in output.

In [1]:
import pandas as pd
import numpy as np

# Load data
cdcr = pd.read_csv('data/cdcr/cdcr_facilities.csv')
hazard = pd.read_csv('data/hazards/heat_air_hazard.csv')
indoor = pd.read_csv('data/cdcr/indoor_outdoor_heat_2025.csv')

print(f'cdcr_facilities rows: {len(cdcr)}')
print(f'heat_air_hazard rows: {len(hazard)}')
print(f'indoor_outdoor_heat rows: {len(indoor)}')

cdcr_facilities rows: 84
heat_air_hazard rows: 357
indoor_outdoor_heat rows: 31


## 1. Build working dataset — 31 CDCR state prisons

In [2]:
# Filter to CDCR state prisons (has cdcr_code, not fire camp)
state_prisons = cdcr[
    cdcr['cdcr_code'].notna() &
    (cdcr['cdcr_firecamp'].fillna(False) != True)
].copy()
print(f'State prisons in cdcr_facilities: {len(state_prisons)}')

# Inner join with indoor_outdoor — this restricts to the 31 with exposure data
# (excludes CAC, CVSP, FWF which have no indoor heat model data)
df = state_prisons.merge(indoor, on='cdcr_code', how='inner', suffixes=('', '_indoor'))
print(f'After join with indoor_outdoor: {len(df)} facilities')
print(f'Facilities: {sorted(df["cdcr_code"].tolist())}')

State prisons in cdcr_facilities: 34
After join with indoor_outdoor: 31 facilities
Facilities: ['ASP', 'CAL', 'CCI', 'CCWF', 'CEN', 'CHCF', 'CIM', 'CIW', 'CMC', 'CMF', 'COR', 'CRC', 'CTF', 'FOL', 'HDSP', 'ISP', 'KVSP', 'LAC', 'MCSP', 'NKSP', 'PBSP', 'PVSP', 'RJD', 'SAC', 'SATF', 'SCC', 'SOL', 'SQ', 'SVSP', 'VSP', 'WSP']


In [3]:
# Join hazard by cdcr_code — v0.2: heat_air_hazard.csv is facility-keyed, so each
# prison gets its own LOCA2-CA cell (replacing the v0.1 tract-centroid join).
# Bring the relative + absolute daytime counts, the relative warm-night count, and AQI;
# the hazard composite is recomputed below, max-normalized across these 31 index facilities.
haz_cols = [
    'cdcr_code',
    'loca2_days_over_avg_plus10_historic', 'loca2_days_over_avg_plus10_midcentury',
    'loca2_days_over_90_historic', 'loca2_days_over_90_midcentury',
    'loca2_nights_over_p95_historic', 'loca2_nights_over_p95_midcentury',
    'AQI_norm',
]
df = df.merge(hazard[haz_cols], on='cdcr_code', how='left')

n_null = df['loca2_days_over_avg_plus10_historic'].isnull().sum()
print(f'Hazard join nulls: {n_null}')
print(f'days_over_avg_plus10 midcentury range: '
      f'{df["loca2_days_over_avg_plus10_midcentury"].min():.1f} – '
      f'{df["loca2_days_over_avg_plus10_midcentury"].max():.1f}')
print(f'days_over_90 midcentury range: '
      f'{df["loca2_days_over_90_midcentury"].min():.1f} – '
      f'{df["loca2_days_over_90_midcentury"].max():.1f}')
print(f'nights_over_p95 midcentury range: '
      f'{df["loca2_nights_over_p95_midcentury"].min():.1f} – '
      f'{df["loca2_nights_over_p95_midcentury"].max():.1f}')
print(f'AQI_norm range: {df["AQI_norm"].min():.1f} – {df["AQI_norm"].max():.1f}')

Hazard join nulls: 0
days_over_avg_plus10 midcentury range: 14.0 – 44.3
days_over_90 midcentury range: 1.7 – 208.4
nights_over_p95 midcentury range: 29.5 – 51.8
AQI_norm range: 12.2 – 94.4


## 2. Normalization helper

In [4]:
def minmax_norm(series):
    """Min-max normalize a series to 0–1 across the 31 facilities."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

## 3. Hazard component (v0.2)

Recomputed here from the facility-level LOCA2-CA counts in `data/hazards/heat_air_hazard.csv`,
max-normalized across these 31 index facilities so the hazard shares the same normalization base
as Exposure and Vulnerability:

1. **Hot days (blended)** — a 50/50 blend of a facility-relative threshold
   (`loca2_days_over_avg_plus10`: days above the facility's mean summer daily-max + 10°F,
   1981–2010 baseline) and an absolute threshold (`loca2_days_over_90`: days over 90°F), each
   max-normalized before blending. The 50% weight (`W_REL`) is a parameter; after normalization
   the 80°F-vs-90°F choice is immaterial (ρ≈0.95).
2. **Warm nights** — `loca2_nights_over_p95`: April–October nights with tmin above the 95th
   percentile of the facility's 1961–1990 April–October distribution (OEHHA convention),
   max-normalized.
3. `temp = (day_blend + night_norm) / 2`. Max-normalization (not min-max) means a facility with
   no exceedances scores 0.
4. Air quality is a multiplicative modifier: `H = temp × (1 + 0.30 × AQI_norm/100)`, ×1.0 at
   AQI = 0. AQI_norm is the CalEnviroScreen 5.0 ozone/PM2.5/diesel percentile mean; missing AQI → ×1.
5. Scaled by the cross-period max of `H` to a 0–1 component score.

See the v0.2 changelog in `analysis/README.md`.

In [5]:
# v0.2 hazard equation, normalized across the 31 index facilities (cross-period):
#   1. daytime = 50/50 blend of a facility-relative threshold (days over the facility's own summer
#      mean +10°F) and an absolute threshold (days over 90°F), each max-normalized first
#   2. night = loca2_nights_over_p95, max-normalized (relative)
#   3. temp = (day_blend + night_norm) / 2
#   4. H = temp × (1 + β·AQI_norm/100)      (AQI multiplicative modifier; missing → ×1)
#   5. scale by cross-period max of H  → hazard component on a 0–1 scale (like E and V)
BETA  = 0.30   # AQI amplification coefficient — a design parameter, capped at +30%
W_REL = 0.50   # daytime relative/absolute blend weight (0.50 = equal); documented design parameter

rel_max = df[['loca2_days_over_avg_plus10_historic',
              'loca2_days_over_avg_plus10_midcentury']].to_numpy().max()
abs_max = df[['loca2_days_over_90_historic',
              'loca2_days_over_90_midcentury']].to_numpy().max()
night_max = df[['loca2_nights_over_p95_historic',
                'loca2_nights_over_p95_midcentury']].to_numpy().max()

def day_blend(suf):
    rel = df[f'loca2_days_over_avg_plus10_{suf}'] / rel_max
    ab  = df[f'loca2_days_over_90_{suf}'] / abs_max
    return W_REL * rel + (1 - W_REL) * ab

day_h, day_m = day_blend('historic'), day_blend('midcentury')
night_h = df['loca2_nights_over_p95_historic']   / night_max
night_m = df['loca2_nights_over_p95_midcentury'] / night_max

temp_h = (day_h + night_h) / 2
temp_m = (day_m + night_m) / 2

# AQI multiplicative modifier (missing AQI → ×1)
modifier = 1 + BETA * (df['AQI_norm'].fillna(0) / 100)
H_h = temp_h * modifier
H_m = temp_m * modifier

# Scale by the cross-period max (max-norm, not min-max): 0 means "no exceedances".
H_max = pd.concat([H_h, H_m]).max()
df['hazard_current']    = H_h / H_max
df['hazard_midcentury'] = H_m / H_max

print(f'Hazard component (0–1): daytime {W_REL:.0%} relative / {1-W_REL:.0%} absolute-90F, '
      f'P95 nights, AQI modifier β={BETA}:')
print(df[['cdcr_code', 'hazard_current', 'hazard_midcentury']]
      .sort_values('hazard_midcentury', ascending=False).round(3).to_string(index=False))

Hazard component (0–1): daytime 50% relative / 50% absolute-90F, P95 nights, AQI modifier β=0.3:
cdcr_code  hazard_current  hazard_midcentury
      CIM           0.371              1.000
      CIW           0.365              0.958
      CRC           0.354              0.929
     SATF           0.356              0.909
      COR           0.356              0.909
     CCWF           0.335              0.885
      VSP           0.336              0.879
     CHCF           0.326              0.872
      LAC           0.319              0.857
      ISP           0.430              0.855
     NKSP           0.348              0.847
     KVSP           0.348              0.839
     HDSP           0.282              0.819
      FOL           0.334              0.808
     MCSP           0.319              0.802
      WSP           0.352              0.793
      CEN           0.402              0.785
      SCC           0.300              0.779
      CCI           0.215              0.768
   

## 4. Exposure component

4 equal-weight sub-components, each min-max normalized 0–1:
1. `days_indoor_above_78f_2025` — direct indoor heat burden
2. `ratio_indoor_to_outdoor` — building thermal amplification
3. `uhi_normalized` — geographic urban heat island (Benz & Burney 2021)
4. `1 - pct_hu_mechanical` — inverted mechanical-AC coverage from the CDCR Air Cooling Pilot Supplemental Report (Jan 2026, as of Dec 2025); high AC = low exposure. Replaces the older, incomplete Reuters `pct_units_refrigeration`.

In [6]:
# Sub-component 1: indoor 78°F days
df['exp_indoor78'] = minmax_norm(df['days_indoor_above_78f_2025'])

# Sub-component 2: ratio indoor/outdoor
# PBSP outlier: ratio = 15.75 vs system max ~2.0 for all others
print('ratio_indoor_to_outdoor — top 5:')
print(df[['cdcr_code', 'ratio_indoor_to_outdoor']]
      .sort_values('ratio_indoor_to_outdoor', ascending=False).head(5).to_string(index=False))
df['exp_ratio'] = minmax_norm(df['ratio_indoor_to_outdoor'])

# Sub-component 3: UHI (already 0–1; impute 2 nulls with system mean)
uhi_nulls = df.loc[df['uhi_normalized'].isnull(), 'cdcr_code'].tolist()
print(f'\nuhi_normalized nulls: {uhi_nulls} — imputed with system mean')
uhi_mean = df['uhi_normalized'].mean()
df['uhi_filled'] = df['uhi_normalized'].fillna(uhi_mean)
df['exp_uhi'] = minmax_norm(df['uhi_filled'])

# Sub-component 4: inverted mechanical-AC fraction.
# Uses pct_hu_mechanical — the fraction of housing units (wings/dorms/tiers)
# with refrigerated/mechanical cooling from the CDCR Air Cooling Pilot Supplemental
# Report (Jan 2026, as of Dec 2025). This REPLACES pct_units_refrigeration (the older
# Reuters FOIA equipment inventory), which was incomplete at 11/31 facilities and gave
# spurious values — e.g. CIM 100% vs the CDCR report's ~43%. Only mechanical counts as
# real AC here; evaporative and ventilation both read as "no AC" = heat-exposed.
df['ac_inverted'] = 1 - df['pct_hu_mechanical']
df['exp_noac'] = minmax_norm(df['ac_inverted'])

# Exposure score = equal-weight mean of 4 sub-components
exp_cols = ['exp_indoor78', 'exp_ratio', 'exp_uhi', 'exp_noac']
df['exposure_score'] = df[exp_cols].mean(axis=1)

print('\nExposure sub-components and score:')
print(df[['cdcr_code'] + exp_cols + ['exposure_score']]
      .sort_values('exposure_score', ascending=False).to_string(index=False))

ratio_indoor_to_outdoor — top 5:
cdcr_code  ratio_indoor_to_outdoor
     PBSP                   15.750
      CTF                    1.935
      RJD                    1.932
       SQ                    1.414
      CCI                    1.278

uhi_normalized nulls: ['PVSP', 'CCI'] — imputed with system mean

Exposure sub-components and score:
cdcr_code  exp_indoor78  exp_ratio  exp_uhi  exp_noac  exposure_score
     PBSP      0.391304   1.000000 0.309600    0.9655        0.666601
      CIM      1.000000   0.063111 1.000000    0.5714        0.658628
      COR      0.987578   0.061905 0.577900    1.0000        0.656846
      SOL      0.987578   0.069143 0.581500    0.8400        0.619555
      CRC      0.925466   0.061016 0.571300    0.9167        0.618620
     NKSP      0.832298   0.052508 0.627600    0.9600        0.618102
     SATF      0.826087   0.051810 0.600800    0.9375        0.604049
      WSP      0.844720   0.052000 0.483900    1.0000        0.595155
      CIW      0.931677  

## 5. Vulnerability component

6 equal-weight sub-components, each min-max normalized 0–1:
1. **Medical acuity** — sum of P1 + P2 + medium CCHCS risk tiers (% of facility population)
2. **Age** — % over 50
3. **Mental health** — % EOP designation
4. **Disability** — % DPP placement
5. **Race/POC** — % people of color (heat inequity + structural vulnerability)
6. **Restricted housing (RHU)** — 12-month average % of facility population in restricted housing units (2025). Unlike the other sub-indicators, this variable captures constrained adaptive capacity rather than physiological susceptibility: RHU residents cannot access cooler areas of the facility, cannot self-regulate their location or activity during heat events, and have out-of-cell time averaging approximately one hour per day. Included in the vulnerability component because adaptive capacity is not treated as a standalone fourth component in this framework; RHU placement is the within-system structural condition that most acutely limits adaptive behavior. Source: CDCR Office of Research, STA429 Monthly Restricted Housing Reports, 2025. See Cloud et al. (2023) for the explicit link between solitary confinement and heat vulnerability.


In [7]:
# Medical acuity = P1 + P2 + medium risk
df['medical_acuity'] = (
    df['cchcs_high_risk_p1_pct_2025'] +
    df['cchcs_high_risk_p2_pct_2025'] +
    df['cchcs_medium_risk_pct_2025']
)

vuln_inputs = {
    'medical_acuity': 'medical_acuity',
    'age_over_50':    'cchcs_age_over_50_pct_2025',
    'mental_health':  'cchcs_mental_health_eop_pct_2025',
    'disability':     'cchcs_dpp_pct_2025',
    'race_poc':       'race_peopleofcolor_pct',
    'gender_female': 'gender_female_pct',
}

# Check for nulls
print('Vulnerability input nulls:')
for label, col in vuln_inputs.items():
    n = df[col].isnull().sum()
    print(f'  {label} ({col}): {n} nulls')

# Normalize each sub-component
vuln_norm_cols = []
for label, col in vuln_inputs.items():
    norm_col = f'vuln_{label}'
    df[norm_col] = minmax_norm(df[col])
    vuln_norm_cols.append(norm_col)

# Vulnerability score = equal-weight mean
df['vulnerability_score'] = df[vuln_norm_cols].mean(axis=1)

print('\nVulnerability sub-components and score:')
print(df[['cdcr_code'] + vuln_norm_cols + ['vulnerability_score']]
      .sort_values('vulnerability_score', ascending=False).to_string(index=False))

Vulnerability input nulls:
  medical_acuity (medical_acuity): 0 nulls
  age_over_50 (cchcs_age_over_50_pct_2025): 0 nulls
  mental_health (cchcs_mental_health_eop_pct_2025): 0 nulls
  disability (cchcs_dpp_pct_2025): 0 nulls
  race_poc (race_peopleofcolor_pct): 0 nulls
  gender_female (gender_female_pct): 0 nulls

Vulnerability sub-components and score:
cdcr_code  vuln_medical_acuity  vuln_age_over_50  vuln_mental_health  vuln_disability  vuln_race_poc  vuln_gender_female  vulnerability_score
     CHCF             1.000000          1.000000            0.504854         1.000000       0.090454            0.000000             0.599218
      CMF             0.931385          0.810507            0.587379         0.777570       0.175051            0.002445             0.547389
      RJD             0.872935          0.652908            0.618932         0.583178       0.363903            0.000000             0.515309
      SAC             0.869123          0.257036            1.000000        

## 6. Risk score

Risk = 0.25 × Hazard + 0.25 × Exposure + 0.50 × Vulnerability

Normalized 0–100 **cross-period by max-normalization** (v0.3): current and mid-century scores are divided by the same cross-period maximum, so they are directly comparable and the current → mid-century increase is preserved. Unlike min-max (v0.2), max-normalization does not subtract the minimum, so no facility is forced to a false 0 — every prison retains a positive score reflecting its real hazard and vulnerability. (This is the same normalization used inside the hazard component, §3.)

In [8]:
H_cur = df['hazard_current']
H_mid = df['hazard_midcentury']
E = df['exposure_score']
V = df['vulnerability_score']

df['raw_risk_current']    = 0.25 * H_cur + 0.25 * E + 0.50 * V
df['raw_risk_midcentury'] = 0.25 * H_mid + 0.25 * E + 0.50 * V

# Cross-period MAX-normalization (v0.3): divide by the cross-period maximum only, with no
# subtraction of the minimum. Min-max (the v0.2 approach) forces the single lowest
# facility-period to exactly 0 — a misleading "zero heat risk" for a prison that still carries
# real hazard and vulnerability. That is why CEN read 0.0 in the current period under v0.2.
# Max-normalization keeps the SHARED cross-period denominator — so the two periods stay directly
# comparable and the current -> mid-century increase is preserved — while removing the false
# floor: the lowest facility now maps to its true (positive) share of the maximum. This mirrors
# the max-normalization already used inside the hazard component (see §3).
all_raw = pd.concat([df['raw_risk_current'], df['raw_risk_midcentury']])
raw_max = all_raw.max()
print(f'Raw risk (both periods): max {raw_max:.4f}, min {all_raw.min():.4f} '
      f'(min maps to {all_raw.min() / raw_max * 100:.1f}, not 0)')

df['risk_score_current']    = df['raw_risk_current']    / raw_max * 100
df['risk_score_midcentury'] = df['raw_risk_midcentury'] / raw_max * 100

print('\nRisk scores — mid-century ranked:')
print(df[['cdcr_code', 'hazard_midcentury', 'exposure_score', 'vulnerability_score',
          'risk_score_current', 'risk_score_midcentury']]
      .sort_values('risk_score_midcentury', ascending=False)
      .round(2).to_string(index=False))

Raw risk (both periods): max 0.6368, min 0.2166 (min maps to 34.0, not 0)

Risk scores — mid-century ranked:
cdcr_code  hazard_midcentury  exposure_score  vulnerability_score  risk_score_current  risk_score_midcentury
      CIM               1.00            0.66                 0.44               75.30                 100.00
      CMF               0.76            0.49                 0.55               74.57                  91.79
      CIW               0.96            0.56                 0.41               68.22                  91.52
      COR               0.91            0.66                 0.38               69.77                  91.50
     SATF               0.91            0.60                 0.40               69.44                  91.17
      SAC               0.73            0.45                 0.51               69.26                  86.85
     CHCF               0.87            0.09                 0.60               63.19                  84.65
     CCWF          

## 7. Risk categories and output

Risk categories derived from Jenks natural breaks (k=4) computed **separately for each time period** (v0.3).
Labels: **Lowest → Moderate → High → Highest**.

Each period is classified against its own range, so both periods span all four categories. A category is therefore relative *within* a period — "Highest" in the current period is a lower absolute risk than "Highest" mid-century. The 0–100 score (§6) stays cross-period normalized, so the absolute increase over time is carried by the score; only the label is within-period. (v0.2 applied the mid-century breaks to both periods, which collapsed the cooler current period into just Lowest/Moderate.)

Long format: two rows per facility (current + mid-century).
Saved to `data/cdcr/CDCR_heat_risk_index_additive_25_25_50.csv`.

In [9]:
import os, shutil
import jenkspy

labels = ['Lowest', 'Moderate', 'High', 'Highest']

def jenks_labeler(scores):
    """Jenks natural breaks (k=4) on THIS period's scores. Returns (label_fn, inner_edges)."""
    edges = jenkspy.jenks_breaks(scores.tolist(), n_classes=4)[1:]  # drop the running min
    def label(score):
        for i, b in enumerate(edges):
            if score <= b:
                return labels[i]
        return labels[-1]
    return label, [round(b, 2) for b in edges]

# Per-period Jenks breaks (v0.3): each time period is classified against its OWN range, so both
# periods use all four labels. Under v0.2 the mid-century breaks were applied to both periods,
# which — because the shared-denominator score makes the cooler current period genuinely lower —
# collapsed the current period into just Lowest/Moderate.
#
# CONSEQUENCE: a category is now RELATIVE WITHIN a period. "Highest" in the current period is a
# lower ABSOLUTE risk than "Highest" mid-century. The 0–100 risk_score stays cross-period
# normalized (§6), so the absolute current -> mid-century increase is still carried by the score;
# only the label is within-period. A facility high on the constant E+V (e.g. RJD, PBSP) can
# therefore sit a band higher in the current period — where hazard spread is compressed — than
# mid-century, even though its score rises.
label_cur, breaks_cur = jenks_labeler(df['risk_score_current'])
label_mid, breaks_mid = jenks_labeler(df['risk_score_midcentury'])
print(f'Jenks breaks — current:     {breaks_cur}')
print(f'Jenks breaks — mid-century: {breaks_mid}')

df['risk_category_current']    = df['risk_score_current'].apply(label_cur)
df['risk_category_midcentury'] = df['risk_score_midcentury'].apply(label_mid)

for per, col in [('current', 'risk_category_current'), ('mid-century', 'risk_category_midcentury')]:
    counts = df[col].value_counts().reindex(labels).to_dict()
    print(f'{per:12s} category counts: {counts}')

print('\nMid-century risk categories:')
print(df[['cdcr_code', 'risk_score_midcentury', 'risk_category_midcentury']]
      .sort_values('risk_score_midcentury', ascending=False).to_string(index=False))

# ── Output ──────────────────────────────────────────────────────────────────
OUT_CSV = 'data/cdcr/CDCR_heat_risk_index_additive_25_25_50.csv'

# Archive the build currently on disk before overwriting it, keyed by its OWN version tag, so
# every shipped version stays retrievable under data/cdcr/archive/. Idempotent (skips if already
# archived). Prior versions (v0.1, v0.2) were moved into this archive dir during the v0.3 cleanup.
ARCHIVE = 'data/cdcr/archive'
os.makedirs(ARCHIVE, exist_ok=True)
if os.path.exists(OUT_CSV):
    _prev_ver = pd.read_csv(OUT_CSV)['index_version'].iloc[0]
    _snap = f'{ARCHIVE}/CDCR_heat_risk_index_additive_25_25_50_{_prev_ver}.csv'
    # Archive only a SUPERSEDED build — skip when the on-disk file is already this version
    # (e.g. a plain re-run), so the archive holds only prior versions and the top-level file is
    # the single active one, never duplicated into the archive.
    if _prev_ver != 'v0.3' and not os.path.exists(_snap):
        shutil.copyfile(OUT_CSV, _snap)
        print(f'Archived superseded build ({_prev_ver}) -> {_snap}')

shared_cols = [
    'cdcr_code', 'name', 'latitude', 'longitude', 'average_2025_population',
    'exposure_score', 'vulnerability_score',
    'AQI_norm', 'ratio_indoor_to_outdoor', 'days_indoor_above_78f_2025',
    'uhi_normalized',
    # Cooling mix (CDCR Air Cooling Pilot Supplemental Report, Jan 2026, as of Dec 2025);
    # pct_hu_mechanical is the AC input to the exposure component.
    'pct_hu_mechanical', 'pct_hu_evaporative', 'pct_hu_air_handlers',
    # vulnerability raw inputs for interpretability
    'medical_acuity', 'cchcs_age_over_50_pct_2025',
    'cchcs_mental_health_eop_pct_2025', 'cchcs_dpp_pct_2025', 'race_peopleofcolor_pct', 'gender_female_pct',
    'rhu_pct_2025',
    # descriptive (not scored)
    'dist_nearest_medical_mi', 'in_urban_area_2020', 'california_model_facility', 'year_opened',
]

current = df[shared_cols + ['hazard_current', 'risk_score_current', 'risk_category_current']].copy()
current = current.rename(columns={
    'hazard_current': 'hazard_score',
    'risk_score_current': 'risk_score',
    'risk_category_current': 'risk_category',
})
current['time_period'] = 'current'

midcentury = df[shared_cols + ['hazard_midcentury', 'risk_score_midcentury', 'risk_category_midcentury']].copy()
midcentury = midcentury.rename(columns={
    'hazard_midcentury': 'hazard_score',
    'risk_score_midcentury': 'risk_score',
    'risk_category_midcentury': 'risk_category',
})
midcentury['time_period'] = 'midcentury'

output = pd.concat([current, midcentury], ignore_index=True)
output = output.sort_values(['cdcr_code', 'time_period']).reset_index(drop=True)

score_cols = ['hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score']
output[score_cols] = output[score_cols].round(2)

# Self-identifying version tag (see analysis/README.md changelog).
output['index_version'] = 'v0.3'

output.to_csv(OUT_CSV, index=False)
print(f'\nSaved {len(output)} rows to {OUT_CSV}')
print(f'Facilities: {output["cdcr_code"].nunique()}, Time periods: {output["time_period"].unique()}, '
      f'version: {output["index_version"].unique()[0]}')
print(f'Columns: {list(output.columns)}')

summary = output[output['time_period'] == 'midcentury'][
    ['cdcr_code', 'hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score', 'risk_category']
].sort_values('risk_score', ascending=False).reset_index(drop=True)
summary.index += 1
print('\nMid-century risk ranking:')
print(summary.to_string())

Jenks breaks — current:     [np.float64(43.23), np.float64(54.21), np.float64(63.19), np.float64(75.3)]
Jenks breaks — mid-century: [np.float64(63.37), np.float64(75.86), np.float64(86.85), np.float64(100.0)]
current      category counts: {'Lowest': 6, 'Moderate': 9, 'High': 9, 'Highest': 7}
mid-century  category counts: {'Lowest': 8, 'Moderate': 10, 'High': 8, 'Highest': 5}

Mid-century risk categories:
cdcr_code  risk_score_midcentury risk_category_midcentury
      CIM             100.000000                  Highest
      CMF              91.789140                  Highest
      CIW              91.520383                  Highest
      COR              91.496449                  Highest
     SATF              91.168052                  Highest
      SAC              86.849215                     High
     CHCF              84.645341                     High
     CCWF              84.382527                     High
      LAC              82.163790                     High
      VSP   

## 8. PBSP ratio outlier check

In [10]:
# PBSP has ratio_indoor_to_outdoor = 15.75 — all other facilities are < 2.0
# Check how much this outlier inflates PBSP's exposure score vs. a capped version

pbsp = df[df['cdcr_code'] == 'PBSP'].iloc[0]
print(f'PBSP ratio: {pbsp["ratio_indoor_to_outdoor"]}')
print(f'PBSP outdoor 78F days (implied): {pbsp["days_indoor_above_78f_2025"] / pbsp["ratio_indoor_to_outdoor"]:.1f}')
print(f'PBSP exposure_score (with outlier): {pbsp["exposure_score"]:.3f}')
print(f'PBSP exp_ratio (with outlier): {pbsp["exp_ratio"]:.3f}')

# What would PBSP exposure score be if ratio were capped at p95 of other facilities?
other_ratios = df.loc[df['cdcr_code'] != 'PBSP', 'ratio_indoor_to_outdoor']
p95 = other_ratios.quantile(0.95)
print(f'\n95th pctl ratio (excl. PBSP): {p95:.3f}')
print('(No cap applied — outlier retained. Consider sensitivity analysis if PBSP rank is influential.)')

PBSP ratio: 15.75
PBSP outdoor 78F days (implied): 4.0
PBSP exposure_score (with outlier): 0.667
PBSP exp_ratio (with outlier): 1.000

95th pctl ratio (excl. PBSP): 1.699
(No cap applied — outlier retained. Consider sensitivity analysis if PBSP rank is influential.)
